# Quickstart: audited SUPPORT2 survival pipeline

This notebook reproduces the paper's core result (Table 2 and Figure 12) in **5 minutes** and provides a copy-paste starting point for researchers building on the audited SUPPORT2 benchmark.

**Paper**: Clevix Lab (2026). *Feature Leakage in a Legacy Clinical Benchmark: A Reproducible Audit of SUPPORT2 Survival Analysis.* PLOS ONE (submitted).

**Repository**: https://github.com/ClevixLab/support2-survival

## What you get

1. The cleaned 9,105-patient SUPPORT2 cohort loaded as a pandas DataFrame.
2. The audited 28-feature admission-time feature matrix plus 3 missing-indicator columns (Table S1).
3. A Gradient Boosting Survival model trained and evaluated on the paper's canonical 70/15/15 split (seed 42).
4. A side-by-side comparison of SHAP feature importance *before* and *after* audit, reproducing Figure 12.

## How to adopt

- Copy the `Layer 1 classification → Layer 2 marginal-C filter → Layer 3 missing-indicator` cells to your own pipeline.
- Swap `support2_full.csv` for your dataset and re-run the audit steps — the framework is dataset-agnostic.
- Start your feature set from `FEATURES_AUDITED` (the list this notebook builds) rather than from raw columns.

## Runtime

~3 minutes on a laptop CPU. No GPU required.

## 0 — Setup

In [ ]:
# If running standalone (not inside the repo), uncomment:
# !pip install -q scikit-survival scikit-learn pandas numpy shap matplotlib lifelines

import sys
from pathlib import Path

# Ensure repo modules are importable when running this notebook from notebooks/
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option("display.max_columns", 50)
print(f"repo path: {REPO}")

## 1 — Load the raw SUPPORT2 dataset

The public SUPPORT2 release is 9,105 ICU/hospital patients × 47 variables from Knaus et al. 1995 (Phase I and Phase II). It is included in the repo at `data/support2_full.csv`. If missing, download it from Harrell's hbiostat server.

In [ ]:
from config import DATA_CSV

df = pd.read_csv(DATA_CSV)
df.columns = df.columns.str.strip()
df["d.time"] = pd.to_numeric(df["d.time"], errors="coerce")
df["death"]  = pd.to_numeric(df["death"],  errors="coerce")
df = df[df["d.time"] > 0].dropna(subset=["d.time", "death", "dzgroup"]).reset_index(drop=True)

print(f"n = {len(df):,} patients, {df.shape[1]} columns")
print(f"event rate = {df['death'].mean():.1%}")
print(f"disease groups: {df['dzgroup'].nunique()}")
print(f"median follow-up: {df['d.time'].median():.0f} days")

## 2 — Apply the three-layer audit

This is the core contribution of the paper. Every variable falls into one of 7 categories:

- **OUTCOME**, **OUTCOME_ALT**: never used as predictors
- **POST-EVENT**: populated after the outcome is observed (leakage)
- **DERIVED**: computed using outcomes or post-admission data (leakage)
- **TREATMENT**: clinical-trial intervention arm (experimental design artifact)
- **REDUNDANT**: partial components of a composite already in the feature set
- **ADMISSION**: legitimate admission-time predictors

We exclude everything except **ADMISSION** (with two clarifications: `hday` is re-included per the original Knaus specification, and three lab features receive missing indicators via Layer 3).

In [ ]:
# These come from the paper's config.py — single source of truth
from config import SKIP_FEATURES, MISSING_INDICATOR_FEATURES

# The full audited pipeline is in data.py — one call does everything:
#   - Layer 1 exclusions
#   - Layer 2 marginal-C empirical check (logged, not re-filtered)
#   - Layer 3 missing-indicator columns
#   - Train/validation/test 70/15/15 split with seed 42
#   - Median imputation fit on TRAINING ONLY
#   - StandardScaler fit on TRAINING ONLY
from data import prepare_data

X_train, X_val, X_test, y_train, y_val, y_test, meta = prepare_data(df)

print(f"Train / Val / Test = {len(X_train)} / {len(X_val)} / {len(X_test)}")
print(f"Final feature count: {X_train.shape[1]}")
print(f"\nFirst 10 audited features: {list(X_train.columns[:10])}")
print(f"\nMissing indicators: {[c for c in X_train.columns if c.endswith('_missing')]}")
print(f"\nExcluded variables (Layer 1): {len(SKIP_FEATURES)}")
for v in SKIP_FEATURES:
    print(f"  - {v}")

**Table S1 is right here**: `list(X_train.columns)` is the audited feature set. Copy this list and use it as your starting feature matrix on any SUPPORT2 analysis.

In [ ]:
# Save as JSON for re-use in other projects
import json
FEATURES_AUDITED = list(X_train.columns)
print(f"Total: {len(FEATURES_AUDITED)} features")
print(json.dumps(FEATURES_AUDITED, indent=2))

## 3 — Train Gradient Boosting Survival

The paper's primary model. Uses scikit-survival's `GradientBoostingSurvivalAnalysis` with published best-practice defaults from the library's documentation (no hyperparameter search — see Methods 2.6).

In [ ]:
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from config import GBM_PARAMS
import time

print(f"GBM hyperparameters: {GBM_PARAMS}")
t0 = time.time()
gbm = GradientBoostingSurvivalAnalysis(**GBM_PARAMS).fit(X_train, y_train)
print(f"Trained in {time.time()-t0:.1f}s")

## 4 — Evaluate on held-out test set (paper Table 2 reproduction)

In [ ]:
from sksurv.metrics import concordance_index_censored, integrated_brier_score, cumulative_dynamic_auc

risk_test = gbm.predict(X_test)
c_index = concordance_index_censored(
    y_test["event"].astype(bool), y_test["time"].astype(float), risk_test
)[0]

# Bootstrap 95% CI
rng = np.random.default_rng(42)
n_test = len(risk_test)
n_boot = 500
boot = []
for _ in range(n_boot):
    idx = rng.integers(0, n_test, n_test)
    try:
        c = concordance_index_censored(
            y_test["event"][idx].astype(bool),
            y_test["time"][idx].astype(float),
            risk_test[idx],
        )[0]
        boot.append(c)
    except Exception:
        pass

ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
print(f"Test C-index = {c_index:.4f}   95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"(Paper reports 0.7052 [0.6861, 0.7230])")

# IBS
t_min = max(y_train["time"].min(), y_test["time"].min()) + 1e-3
t_max = min(y_test["time"].max(), np.percentile(y_train["time"], 95)) - 1e-3
times_grid = np.linspace(t_min, t_max, 50)
surv_fns = gbm.predict_survival_function(X_test)
S = np.asarray([[fn(t) for t in times_grid] for fn in surv_fns])
ibs = integrated_brier_score(y_train, y_test, S, times_grid)
print(f"\nIntegrated Brier Score = {ibs:.4f}   (paper: 0.178)")

## 5 — Reproduce Figure 12: SHAP importance *before* vs *after* audit

This is the headline empirical demonstration of the paper. The post-discharge variable `slos` ranks #2 globally in the unaudited matrix; removing it via Layer 1 eliminates the leakage and surfaces clinically interpretable features (neurological status, cancer status, functional reserve, age).

In [ ]:
# Rebuild a BEFORE-audit feature matrix by re-including slos
# (using the pipeline's same split + preprocessing so the comparison is clean)

# Quick rebuild: take training rows, add slos as a column, re-standardise.
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

def _rebuild_with_slos(df_raw, idx_train, idx_test):
    """Make train/test matrices that include slos as a feature."""
    cols_keep = list(X_train.columns) + ["slos"]
    # Merge in slos from raw frame on the same indices
    slos_train = df_raw.loc[idx_train, "slos"].values
    slos_test  = df_raw.loc[idx_test,  "slos"].values
    Xtr = X_train.copy(); Xtr["slos"] = slos_train
    Xte = X_test.copy();  Xte["slos"] = slos_test
    # Median-impute + standardise slos using training only
    imp = SimpleImputer(strategy="median"); sc = StandardScaler()
    Xtr[["slos"]] = sc.fit_transform(imp.fit_transform(Xtr[["slos"]]))
    Xte[["slos"]] = sc.transform(imp.transform(Xte[["slos"]]))
    return Xtr, Xte

idx_train = meta["idx_train"] if "idx_train" in meta else X_train.index
idx_test  = meta["idx_test"]  if "idx_test"  in meta else X_test.index

Xtr_before, Xte_before = _rebuild_with_slos(df, idx_train, idx_test)
print(f"BEFORE-audit matrix: {Xtr_before.shape} (includes slos)")
print(f"AFTER-audit matrix : {X_train.shape}   (no slos)")

In [ ]:
# Train two models
gbm_before = GradientBoostingSurvivalAnalysis(**GBM_PARAMS).fit(Xtr_before, y_train)
gbm_after  = gbm   # already trained in section 3

# Compute SHAP on a test subsample (paper uses 500)
import shap
SUBSAMPLE = 500
rng = np.random.default_rng(42)
samp = rng.choice(len(Xte_before), size=min(SUBSAMPLE, len(Xte_before)), replace=False)

# Use KernelExplainer because GBM survival functions don't expose tree structure via sksurv
# For speed in the quickstart, use a small background + the sample — paper uses full test set
print("Computing SHAP (BEFORE audit) — ~30s...")
expl_before = shap.Explainer(gbm_before.predict, Xtr_before.iloc[:100])
sv_before = expl_before(Xte_before.iloc[samp])

print("Computing SHAP (AFTER audit)  — ~30s...")
expl_after = shap.Explainer(gbm_after.predict, X_train.iloc[:100])
sv_after = expl_after(X_test.iloc[samp])

# Mean absolute SHAP per feature
importance_before = pd.Series(np.abs(sv_before.values).mean(axis=0),
                              index=Xte_before.columns).sort_values(ascending=False)
importance_after  = pd.Series(np.abs(sv_after.values).mean(axis=0),
                              index=X_test.columns).sort_values(ascending=False)

print("\nTop-5 BEFORE audit:")
print(importance_before.head())
print("\nTop-5 AFTER audit:")
print(importance_after.head())

In [ ]:
# Figure 12 reproduction — side-by-side bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

top_before = importance_before.head(10)
top_after  = importance_after.head(10)

colors_before = ["#D85A30" if f == "slos" else "#888888" for f in top_before.index]
ax1.barh(range(len(top_before)), top_before.values[::-1], color=colors_before[::-1])
ax1.set_yticks(range(len(top_before)))
ax1.set_yticklabels(top_before.index[::-1])
ax1.set_xlabel("mean |SHAP|")
ax1.set_title("BEFORE audit (slos included)")

ax2.barh(range(len(top_after)), top_after.values[::-1], color="#1D9E75")
ax2.set_yticks(range(len(top_after)))
ax2.set_yticklabels(top_after.index[::-1])
ax2.set_xlabel("mean |SHAP|")
ax2.set_title("AFTER audit (slos removed)")

plt.suptitle("Figure 12 reproduction — SHAP feature importance", fontsize=13)
plt.tight_layout()
plt.show()

## 6 — What next?

### For researchers building on SUPPORT2

- Use `FEATURES_AUDITED` as your starting feature set.
- When adding new features, apply the three-layer audit (Section 4.7 of the paper): check Knaus 1995 specification, compute marginal C-index, check for informative missingness.
- Report subgroup transportability (our Section 3.6 / Table 4) alongside any primary-split C-index you publish.

### For researchers extending the audit to new datasets

Apply the 4-step protocol from the paper's Future Directions:

1. **Layer 1**: classify every variable against the original study data dictionary.
2. **Layer 2**: compute marginal C-index per variable; flag those above your dataset's empirical threshold (we use 0.60 for SUPPORT2's 68% event rate).
3. **Layer 3**: test for informative missingness; add missing-indicator columns where warranted.
4. Report leave-one-group-out (or equivalent internal-external) transportability benchmarks.

Candidate datasets with similar structural issues: MIMIC-IV, eICU, AmsterdamUMCdb, HiRID.

### For reproducibility

Run the full pipeline via `python run_pipeline.py` from the repo root — this reproduces every number in the paper, including all figures, with per-run manifests containing the data SHA-256 hash and config snapshot.

## Citation

If you use this pipeline or the audited feature set in your research, please cite:

> Clevix Lab. Feature Leakage in a Legacy Clinical Benchmark: A Reproducible Audit of SUPPORT2 Survival Analysis. PLOS ONE (submitted, 2026). Code archive: [Zenodo DOI pending].